# End of week 1 exercise


In [ ]:
import os

from openai import OpenAI

In [ ]:
MODEL_COHERE = "command-a-03-2025"
MODEL_OLLAMA = "phi3"

COHERE_BASE_URL = "https://api.cohere.ai/compatibility/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
cohere_api_key = os.getenv("COHERE_API_KEY", "").strip()
if cohere_api_key:
    print("COHERE_API_KEY is set — cloud explanations are available.")
else:
    print("Set COHERE_API_KEY in your environment to use Cohere.")

cohere_client = (
    OpenAI(base_url=COHERE_BASE_URL, api_key=cohere_api_key)
    if cohere_api_key
    else None
)
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
system_prompt = """You are a patient software engineer. Explain technical ideas clearly and concisely.
After the explanation, you may include one or two short multiple-choice questions to check understanding."""


def messages_for(question: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question.strip()},
    ]

In [ ]:
def explain_with_cohere(question: str, *, stream: bool = True) -> str:
    """Call Cohere via OpenAI-compatible chat completions. Optionally stream tokens to stdout."""
    if cohere_client is None:
        raise RuntimeError("COHERE_API_KEY is not set.")
    if stream:
        completion_stream = cohere_client.chat.completions.create(
            model=MODEL_COHERE,
            messages=messages_for(question),
            stream=True,
        )
        parts: list[str] = []
        for chunk in completion_stream:
            delta = chunk.choices[0].delta
            if delta and delta.content:
                print(delta.content, end="", flush=True)
                parts.append(delta.content)
        print()
        return "".join(parts)
    response = cohere_client.chat.completions.create(
        model=MODEL_COHERE,
        messages=messages_for(question),
        stream=False,
    )
    text = response.choices[0].message.content or ""
    print(text)
    return text


def explain_with_ollama(question: str) -> str:
    """Local explanation via Ollama (OpenAI-compatible server)."""
    response = ollama_client.chat.completions.create(
        model=MODEL_OLLAMA,
        messages=messages_for(question),
    )
    return (response.choices[0].message.content or "").strip()


def feedback_on_quiz_answer(
    question: str,
    assistant_explanation: str,
    student_reply: str,
    *,
    use_ollama: bool = True,
) -> str:
    """Optional: send your answers to the MC questions back for gentle feedback."""
    grader_system = (
        "You briefly check the user's answers to your prior multiple-choice questions. "
        "Say 'Well done!' when correct; if wrong, correct gently for a beginner."
    )
    msgs = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question.strip()},
        {"role": "assistant", "content": assistant_explanation},
        {"role": "system", "content": grader_system},
        {"role": "user", "content": f"My answers: {student_reply.strip()}"},
    ]
    client = ollama_client if use_ollama else cohere_client
    if client is None:
        raise RuntimeError("Pick use_ollama=True or set COHERE_API_KEY for use_ollama=False.")
    model = MODEL_OLLAMA if use_ollama else MODEL_COHERE
    response = client.chat.completions.create(model=model, messages=msgs)
    return (response.choices[0].message.content or "").strip()

In [ ]:
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
cohere_answer = explain_with_cohere(question)

In [ ]:
ollama_answer = explain_with_ollama(question)
print(ollama_answer)